# SMETANA and SteadyCom

In [1]:
import reframed
from reframed import SteadyCom
import pandas as pd
import numpy as np
from os import listdir
import warnings
import seaborn as sns
from collections import OrderedDict
import json

import subprocess
import time

#### Load data and models

**Load relative abundance (E. coli)**

In [2]:
relative_abundance_ecoli = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance_ecoli.index = relative_abundance_ecoli.index.map(lambda x:x if x!="yhhK" else "panZ")

**Load models**

*E. coli mutants  (top 7)*

In [3]:
RA_ecoli_mean = relative_abundance_ecoli.T.mean().sort_values(ascending=False)
top7 = RA_ecoli_mean[RA_ecoli_mean>0.01].index
top7_string = [f"mutant_{mutant}.xml" for mutant in top7]

In [4]:
models_ecoli = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Guillen2021/strains/"):
        if path not in top7_string: continue
        models_ecoli[path.split("_")[1][:-4]] = reframed.load_cbmodel("../models/Guillen2021/strains/"+path,
                                                        exchange_detection="^R_EX_(?!nad_c$|thf_c$).*")

*Gut microbiome members*

In [5]:
models_gut = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Venturelli2018/species_used_noO2/"):
        models_gut[path[:-4]] = reframed.load_cbmodel("../models/Venturelli2018/species_used_noO2/"+path,
                                                      exchange_detection="R_EX_(?!biomass__40__e__41__)")


**Load media**

*For E. coli mutants (M9 medium to force growth of specific members)*

In [6]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="M9"].compound.values)
env_ecoli = reframed.Environment.from_compounds(compounds,max_uptake=10)

*For gut microbiome members*

In [7]:
media_db_gut = pd.read_csv("../data/Venturelli2018/generated/media_db.tsv",sep="\t")
reactions = media_db_gut[media_db_gut.medium=="LB_extend"].compound.values
#reactions = [rxn.replace("40","91").replace("41","93") for rxn in reactions]

env_gut = reframed.Environment.from_reactions(reactions,max_uptake=10)

**Load other data**

In [8]:
gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t")

## SMETANA

### Prepare files for SMETANA 

We need:
- Community models
- Community data file

We also need:
- Media



**E.coli mutants**

*Prepare files*

**Gut microbiome**

*Prepare files*

### Run SMETANA

## SteadyCom

In [9]:
reframed.set_default_solver("cplex")

### Test: Run SteadyCom for E.coli with fixed abundance

In [10]:
relative_abundance_exp = pd.read_csv("../data/Guillen2021/generated/communities_steadiercom.tsv",sep="\t",header=None)
relative_abundance_exp[2] = relative_abundance_exp[2]/relative_abundance_exp[2].sum()
relative_abundance_exp

,0,1,2
0,Ecoli_mutants,mutant_icd,0.295973
1,Ecoli_mutants,mutant_panZ,0.284488
2,Ecoli_mutants,mutant_bioH,0.217089
3,Ecoli_mutants,mutant_bioF,0.101307
4,Ecoli_mutants,mutant_cysW,0.063975
5,Ecoli_mutants,mutant_bioC,0.020770
6,Ecoli_mutants,mutant_panD,0.016398


In [11]:
# Prepare for SteadyCom
constraints = {"x_"+row[1]:row[2] for i,row in relative_abundance_exp.iterrows()} # Define constraints for abundance
comm_ecoli = reframed.Community(community_id="E.coli",models=models_ecoli.values())
env_ecoli.apply(comm_ecoli.merged_model,inplace=True,exclusive=True)

# Run SteadyCom
sol_steadycom_ecoli = SteadyCom(comm_ecoli,constraints=constraints)

# Process results
cross_feeding_ecoli = pd.DataFrame(sol_steadycom_ecoli.cross_feeding())
cross_feeding_ecoli.columns = ["donor","receiver","compound","rate?"]

relative_abundance_ecoli = {"relative_abundance":sol_steadycom_ecoli.abundance}
growth_ecoli = {"growth":{"E.coli mutants":sol_steadycom_ecoli.growth}}

# Save results
pd.DataFrame(growth_ecoli).to_csv("../output/Guillen2021/steadycom/growth_fixed_abundance.tsv",sep="\t")
pd.DataFrame(relative_abundance_ecoli).to_csv("../output/Guillen2021/steadycom/relative_abundance_fixed_abundance.tsv",sep="\t")
pd.DataFrame(cross_feeding_ecoli).to_csv("../output/Guillen2021/steadycom/cross_feeding_fixed_abundance.tsv",sep="\t",index=None)

### Run SteadyCom for E.coli 

In [12]:
# Prepare for SteadyCom
comm_ecoli = reframed.Community(community_id="E.coli",models=models_ecoli.values())
env_ecoli.apply(comm_ecoli.merged_model,inplace=True,exclusive=True)

# Run SteadyCom
sol_steadycom_ecoli = SteadyCom(comm_ecoli)

# Process results
cross_feeding_ecoli = pd.DataFrame(sol_steadycom_ecoli.cross_feeding())
cross_feeding_ecoli.columns = ["donor","receiver","compound","rate?"]

relative_abundance_ecoli = {"relative_abundance":sol_steadycom_ecoli.abundance}
growth_ecoli = {"growth":{"E.coli mutants":sol_steadycom_ecoli.growth}}

# Save results
pd.DataFrame(growth_ecoli).to_csv("../output/Guillen2021/steadycom/growth.tsv",sep="\t")
pd.DataFrame(relative_abundance_ecoli).to_csv("../output/Guillen2021/steadycom/relative_abundance.tsv",sep="\t")
pd.DataFrame(cross_feeding_ecoli).to_csv("../output/Guillen2021/steadycom/cross_feeding.tsv",sep="\t",index=None)

### Run SteadyCom for Gut microbiome

In [13]:
constraints_gut ={}
for file in listdir("../data/Venturelli2018/generated/communities_steadiercom/"):                
    if not file.endswith(".tsv"):
        continue
    
    relative_abundance_exp_gut = pd.read_csv("../data/Venturelli2018/generated/communities_steadiercom/"+file,sep="\t",header=None)
    # make sure the relative abundance sums to 1
    extracted = pd.Series({"x_M_"+row[1]:row[2] for i,row in relative_abundance_exp_gut.iterrows()})
    extracted = extracted/extracted.sum()
    constraints_gut[relative_abundance_exp_gut.iloc[0,0]] =extracted.to_dict()

In [14]:
def SteadyComGutCommunities(fixed_relative_abundance=True):    
    cross_feeding_gut_comms = {}
    relative_abundance_gut_comms = {}
    growth_gut_comms = {}

    for community_id,relative_abundance_dict in constraints_gut.items():
        print(f"Started SteadyCom analysis for {community_id}")
        # Prepare for SteadyCom
        model_ids = [id_[4:] for id_ in relative_abundance_dict.keys()]
        models_specific = [model for id_, model in models_gut.items() if id_ in model_ids]
        
        comm_gut = reframed.Community(community_id=community_id,models=models_specific)
        env_gut.apply(comm_gut.merged_model,inplace=True,exclusive=True,warning=False)
        
        # Run SteadyCom
        time_1 = time.time()
        if fixed_relative_abundance:
            sol_steadycom_gut = SteadyCom(comm_gut,constraints=relative_abundance_dict)
        else:
            sol_steadycom_gut = SteadyCom(comm_gut)
        time_2 = time.time()
        print(f"\tRunning SteadyCom took: {(time_2 - time_1)/60:.2f} minutes")
        
        # Gather data
        cross_feeding_gut = pd.DataFrame(sol_steadycom_gut.cross_feeding())
        cross_feeding_gut.columns = ["donor","receiver","compound","rate?"]
        relative_abundance_gut = sol_steadycom_gut.abundance
        growth_gut = sol_steadycom_gut.growth
        
        cross_feeding_gut_comms[community_id] = cross_feeding_gut
        relative_abundance_gut_comms[community_id] = relative_abundance_gut
        growth_gut_comms[community_id] = growth_gut
        
        time_3 = time.time()
        print(f"\tCollecting results took: {(time_3 - time_2)/60:.2f} minutes")
    return cross_feeding_gut_comms,relative_abundance_gut_comms,growth_gut_comms

#### Fixed abundance

In [15]:
cross_feeding_gut_comms_fixed,relative_abundance_gut_comms_fixed,growth_gut_comms_fixed = SteadyComGutCommunities(fixed_relative_abundance=True)


Started SteadyCom analysis for ER-
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for DP-
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for PC-
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for EL-
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for BT-
	Running SteadyCom took: 0.02 minutes
Started SteadyCom analysis for BU-
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for NONE
	Running SteadyCom took: 1.22 minutes
Started SteadyCom analysis for BV-
	Running SteadyCom took: 0.03 minutes
Started SteadyCom analysis for CH-
	Running SteadyCom took: 0.37 minutes
Started SteadyCom analysis for BO-
	Running SteadyCom took: 0.16 minutes
Started SteadyCom analysis for BH-
	Running SteadyCom took: 0.26 minutes
Started SteadyCom analysis for FP-
	Running SteadyCom took: 0.23 minutes
Started SteadyCom analysis for CA-
	Running SteadyCom took: 0.21 minutes


In [16]:
pd.DataFrame({"growth":growth_gut_comms_fixed}).to_csv("../output/Venturelli2018/steadycom/growth_fixed_abundance.tsv",sep="\t")

pd.DataFrame(relative_abundance_gut_comms_fixed).to_csv("../output/Venturelli2018/steadycom/relative_abundance_fixed_abundance.tsv",sep="\t")

cross_feeding_gut_df = pd.concat(cross_feeding_gut_comms_fixed).reset_index().drop("level_1",axis=1)
cross_feeding_gut_df.columns= ['community', 'donor', 'receiver', 'compound', 'rate?']
cross_feeding_gut_df.to_csv("../output/Venturelli2018/steadycom/cross_feeding_fixed_abundance.tsv",sep="\t",index=None)

#### Not fixed abundance

In [17]:
cross_feeding_gut_comms,relative_abundance_gut_comms,growth_gut_comms = SteadyComGutCommunities(fixed_relative_abundance=False)


Started SteadyCom analysis for ER-
	Running SteadyCom took: 0.47 minutes
Started SteadyCom analysis for DP-
	Running SteadyCom took: 0.39 minutes
Started SteadyCom analysis for PC-
	Running SteadyCom took: 0.51 minutes
Started SteadyCom analysis for EL-
	Running SteadyCom took: 0.50 minutes
Started SteadyCom analysis for BT-
	Running SteadyCom took: 0.51 minutes
Started SteadyCom analysis for BU-
	Running SteadyCom took: 0.49 minutes
Started SteadyCom analysis for NONE
	Running SteadyCom took: 0.72 minutes
Started SteadyCom analysis for BV-
	Running SteadyCom took: 0.37 minutes
Started SteadyCom analysis for CH-
	Running SteadyCom took: 0.61 minutes
Started SteadyCom analysis for BO-
	Running SteadyCom took: 0.82 minutes
Started SteadyCom analysis for BH-
	Running SteadyCom took: 0.80 minutes
Started SteadyCom analysis for FP-
	Running SteadyCom took: 4.68 minutes
Started SteadyCom analysis for CA-
	Running SteadyCom took: 3.59 minutes


In [18]:
pd.DataFrame({"growth":growth_gut_comms}).to_csv("../output/Venturelli2018/steadycom/growth.tsv",sep="\t")

pd.DataFrame(relative_abundance_gut_comms).to_csv("../output/Venturelli2018/steadycom/relative_abundance.tsv",sep="\t")

cross_feeding_gut_df = pd.concat(cross_feeding_gut_comms).reset_index().drop("level_1",axis=1)
cross_feeding_gut_df.columns= ['community', 'donor', 'receiver', 'compound', 'rate?']
cross_feeding_gut_df.to_csv("../output/Venturelli2018/steadycom/cross_feeding.tsv",sep="\t",index=None)